# 06 - NewsQA 200/11064 dataset EDA

Reproduces every number and figure in `docs/eda_report.md`.

**Run all.** With `RECOMPUTE = False` (the default) each section reads the saved
result for its script and prints it, which takes about a minute end to end. Set
`RECOMPUTE = True` to run the analysis scripts themselves from the raw data -
that takes roughly 45 minutes, most of it in the two sections that read all
92,579 archived pages.

The scripts under `scripts/eda/` are the source of truth. This notebook drives
them and narrates the output, so a number here and a number in the report cannot
disagree.

Read-only over `data/evaluation/newsqa_200_11064/`. Nothing here writes to the
locked benchmark.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

# False -> read saved results (about a minute).
# True  -> re-run every analysis from the raw data (about 45 minutes).
RECOMPUTE = False

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'pyproject.toml').exists())
EDA = PROJECT_ROOT / 'scripts' / 'eda'
OUT = EDA / 'out'
PYTHON = PROJECT_ROOT / ('.venv/Scripts/python.exe' if sys.platform == 'win32'
                         else '.venv/bin/python')
if not PYTHON.exists():
    PYTHON = Path(sys.executable)


def run(script: str, produces: str | list[str] | None = None) -> None:
    """Run an EDA script, or print its saved result when RECOMPUTE is False."""
    names = [produces] if isinstance(produces, str) else (produces or [])
    saved = [OUT / f'{n}.json' for n in names]
    if not RECOMPUTE and names and all(p.exists() for p in saved):
        for path in saved:
            print(f'--- {path.name} (saved; set RECOMPUTE = True to redo) ---')
            print(json.dumps(summarise(json.loads(path.read_text('utf-8'))), indent=1))
        return
    process = subprocess.Popen(
        [str(PYTHON), str(EDA / script)], cwd=EDA,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1)
    for line in process.stdout:
        print(line, end='')
    if process.wait() != 0:
        raise RuntimeError(f'{script} failed')


def summarise(payload, limit: int = 12):
    """Drop the long per-item arrays so a saved result prints readably."""
    if isinstance(payload, dict):
        return {k: summarise(v, limit) for k, v in payload.items()
                if not (isinstance(v, list) and len(v) > limit)}
    return payload


def cached(name: str) -> dict:
    return json.loads((OUT / f'{name}.json').read_text(encoding='utf-8'))


print('project :', PROJECT_ROOT)
print('python  :', PYTHON.name)
print('mode    :', 'RECOMPUTE from raw data' if RECOMPUTE else 'read saved results')
print('saved   :', len(list(OUT.glob('*.json'))), 'results available')

## 1. Inventory, profile, cleanliness, consistency

How many articles and questions, how long they are, how many chunks they make,
what is mechanically dirty, and whether the ground truth is internally
consistent - answer positions resolve, gold chunk IDs exist, variant ID sets
agree.

Two things to watch. **1.74 chunks per article** means chunk retrieval is almost
article retrieval here, which caps what a chunking experiment can show. And the
resolved questions are nearly twice as long as the originals, 11 words against
6 - section 6 measures what that buys.

In [ ]:
run('01_profile.py', produces='01_profile')

## 2. Are the articles cut off?

The obvious argument does not work: *"the longest article is just under 4,600
characters, so there must be a limit"*. If there were a hard character limit,
articles would pile up at it. They do not.

What does hold is the sentence-ending gradient - the share of articles ending on
a full stop falls from 88.4% in the shortest quartile to 14.7% in the longest.
Stopping mid-sentence more often the longer an article gets is what a cut-off
looks like.

In [ ]:
run('01d_truncation.py', produces='01d_truncation')

### 2b. Direct comparison with the original pages

Of the articles that can be paired with their original CNN page, our version is
an exact character-for-character prefix in half of them - the beginning of the
real article, then it stops.

`01n_threshold_validity.py` then tests whether a simple length rule could stand
in for that comparison. It cannot: 40% of genuinely truncated articles are
shorter than 3,800 characters.

In [ ]:
run('01e_truncation_proof.py', produces='01e_truncation_proof')
run('01n_threshold_validity.py', produces='01n_threshold_validity')

### 2c. Does it actually hurt the scores?

Far less than the raw rate suggests. NewsQA answers are front-loaded - the
median answer sits at the 18th percentile of article length, and truncation
removes the end. The two barely overlap.

In [ ]:
run('01m_truncation_impact.py', produces='01m_truncation_impact')

## 3. How much text is missing, and can it be restored?

`data/cnn_downloads.tgz` holds 92,579 archived CNN pages - the pages the
benchmark was built from, already in this repository. This section pairs each
benchmark article with its page using the project's own extractor
(`NewsCleaner`) and measures the gap.

**This is the expensive section.** With `RECOMPUTE = True` it reads all 92,579
pages twice, about 30 minutes for the two cells together. Note that the archive
is read rather than the loose HTML files: they are on an SSD, but real-time
antivirus scans every file open and drops throughput to 0.3 MB/s, which turns a
30-second read into 75 minutes.

In [ ]:
run('08_truncation_gap.py', produces='08_truncation_gap')

### 3b. Articles whose page starts differently

Matching on an article's opening fails whenever the page begins with something
we do not have - a promo box, a dateline. This pass anchors on a 100-character
run taken from the middle instead, and finds 375 articles the opening-anchor
structurally cannot reach, lifting coverage from 85.6% to about 89%.

In [ ]:
run('10_pair_rescue.py', produces='10_pair_rescue')

### 3c. Choosing the anchor window by measurement

The window length is not a guess. This measures, across articles where both
versions are available, how long a run of text they share and what share of
articles each candidate window length can find.

In [ ]:
run('11_window_calibration.py', produces='11_window_calibration')

## 4. What was wrong with the questions

Each question carries reason codes - short labels for what was wrong with it.
Every code names something checkable against the article (is a subject stated?
does "it" refer to something named?), never a judgement about difficulty. That
is what makes the repair auditable.

The second table is the control: questions with no defect code gained zero
words, which is what shows the repair was not a blanket rewrite.

In [ ]:
run('05_reason_codes.py', produces='05_reason_codes')

## 5. Question forms and answer types

Checks whether one question shape dominates enough that a single headline metric
would describe that shape rather than the system.

In [ ]:
run('02_question_forms.py', produces='02_question_forms')

## 6. Lexical overlap - original against resolved

Measures how much a question shares with its own correct chunk, against a
random-chunk baseline. This predicts sparse retrieval's advantage directly from
the data, without appealing to any tournament result.

It is also the caveat: repair raises rare-term anchors from 0.33 to 0.89 per
question, and rare terms are what **word-matching retrieval feeds on**. A
sparse-against-dense comparison run only on `resolved` is tilted toward sparse,
which is why both versions have to be reported.

In [ ]:
run('03_lexical_overlap.py', produces='03_lexical_overlap')

## 7. Competition, and whether one chunk is really the only answer

**Competition:** rare terms narrow 19,263 chunks to a median of 20 competitors,
and only 31% of questions reach 10 or fewer. First-stage retrieval gets close
and cannot finish - the dataset-level case for a reranker. Meanwhile 37% of
questions contain no rare term at all, which is where dense retrieval has to
carry the load.

**Unlabelled answers:** how often a non-gold article contains the gold answer.
A raw string match says 23.1%, but string presence is not answering, so matches
are graded by how many of the question's rare terms the distractor also shares.
About 6.5% are strong - the same news event covered twice. That is the
false-negative floor under every reported score.

In [ ]:
run('04_distractor_collision.py', produces='04_distractor_collision')

## 8. Near-duplicate questions

The finding that settles the original-against-resolved question.

In the **original** set, 34 pairs of near-identical questions point at different
articles - *"what does faa say"* appears three times with three different
correct articles. No retriever can separate those; they are unscoreable rather
than hard. Repair eliminates the class entirely, 34 to 0.

The cost shows up in the same output: repair collapsed 47 groups of distinct
questions into word-for-word duplicates, so 1,287 of the 1,336 questions are
actually distinct.

In [ ]:
run('06_near_duplicates.py', produces=['06_near_duplicates_original',
                                       '06_near_duplicates_resolved'])

## 9. Cleaning

`scripts/clean_corpus.py` removes surviving page furniture - video teasers, the
share widget, publisher footers, block-boundary newline runs - into a parallel
`cleaned/` tree. The locked benchmark under `final/` is never touched.

Answer positions are character offsets, so every deletion is recorded in an
offset map and each position is remapped and then re-verified against the
cleaned text. A position that cannot be resolved is flagged, never dropped.

This cell is a dry run; add `--apply` to write.

In [ ]:
print(subprocess.run([str(PYTHON), 'scripts/clean_corpus.py'],
                     cwd=PROJECT_ROOT, capture_output=True, text=True,
                     encoding='utf-8', errors='replace').stdout)

## 10. Figures

Renders the seven figures in `docs/eda_report.md` from the saved results, so a
number in the report and a number in a chart cannot drift apart. 300 DPI PNGs
land in `docs/figures/eda/`.

Colours are the first three slots of a colour-vision-deficiency-validated
palette, assigned by entity (original against resolved) rather than by rank,
with direct value labels on every bar.

In [ ]:
run('07_figures.py')

from IPython.display import Image, display
for name in ('fig1_truncation', 'fig2_evidence_position',
             'fig3_question_repair', 'fig4_retrieval_difficulty',
             'fig5_restoration', 'fig6_truncation_gap',
             'fig7_window_calibration'):
    display(Image(filename=str(PROJECT_ROOT / 'docs' / 'figures' / 'eda'
                               / f'{name}.png')))

## 11. Summary

The headline numbers, pulled from the saved results so this table and the
report's TL;DR cannot disagree.

In [ ]:
rows = []
profile = cached('01_profile')['inventory']
rows.append(('corpus', f"{profile['evaluation articles']} evaluation "
                       f"+ {profile['distractor articles']} distractor"))
rows.append(('chunks @512/64', profile['chunks @512/64']))
rows.append(('questions (resolved)', profile['testset - resolved']))

truncation = cached('01d_truncation')['ends_cleanly']
rows.append(('ends on a sentence, shortest quartile', f"{truncation['shortest 25%']:.1%}"))
rows.append(('ends on a sentence, longest quartile', f"{truncation['longest 25%']:.1%}"))

gap = cached('08_truncation_gap')
rows.append(('articles paired with their original page',
             f"{gap['matched']:,} of {gap['corpus']:,}"))
rows.append(('missing real article text',
             f"{gap['real_content']:,} ({gap['real_content']/gap['matched']:.1%} of paired)"))
rows.append(('typical loss where text is missing',
             f"{gap['real_gap_chars']['median']:,} chars "
             f"({gap['real_gap_percent']['median']}% of the article)"))
rows.append(('found only by the mid-article anchor',
             cached('10_pair_rescue')['matched_with_different_start']))

overlap = cached('03_lexical_overlap')
rows.append(('rare anchors per question, original',
             f"{overlap['original']['rare_gold_mean']:.2f}"))
rows.append(('rare anchors per question, resolved',
             f"{overlap['resolved']['rare_gold_mean']:.2f}"))

collision = cached('04_distractor_collision')
rows.append(('median non-gold competitors', collision['competitors_median']))
rows.append(('unlabelled answers, strong / upper bound',
             f"{collision['strong_candidates']/collision['checked']:.1%}"
             f" / {collision['distractor_has_answer']/collision['checked']:.1%}"))

rows.append(('unscoreable question pairs, original -> resolved',
             f"{cached('06_near_duplicates_original')['cross_article_conflicts']}"
             f" -> {cached('06_near_duplicates_resolved')['cross_article_conflicts']}"))
rows.append(('effective distinct questions',
             f"{1336 - cached('06_convergence')['surplus']:,} of 1,336"))

width = max(len(str(k)) for k, _ in rows)
for key, value in rows:
    print(f'{key:<{width}}  {value}')